# Chapter 2 — Working with text data

Use this notebook to follow along with Chapter 2. Add your notes, experiments, and implementations below as you work through the chapter.

In [ ]:
#| default_exp data

## 2.1 Reading a text corpus

Language models learn from text corpora. This chapter uses Edith Wharton's short story *The Verdict* as a small, manageable dataset.

The next cell downloads the UTF-8 text file from the book's companion repository and saves it beside this notebook. Once downloaded, the local copy can be reused without another network request.

In [1]:
# Download the chapter's sample corpus to the notebook directory.
import urllib.request

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x26db3410e30>)

### Inspecting the raw text

Before tokenizing, load the entire file into one string and inspect its size and opening characters. Looking at a sample helps catch encoding, path, or formatting problems early.

`raw_text` is the source sequence from which the tokenizer will build discrete tokens.

In [2]:
# Read the complete UTF-8 corpus as one string.
with open("the-verdict.txt", encoding="utf-8") as f:
    raw_text = f.read()

# Inspect the corpus size and a short sample before preprocessing.
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2.2 Tokenizing text

A tokenizer converts a string into smaller units called **tokens**. Tokens may be words, punctuation marks, subwords, or special symbols.

We begin with a deliberately simple regular-expression tokenizer. The capturing group in `r'(\s)'` keeps matched whitespace in the result, making each split visible while we develop the rule.

In [3]:
import re

# Start by splitting on whitespace while retaining the separators.
text = "Hello, world. This, is a test."
result = re.split(r"(\s)", text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


### Keeping punctuation as tokens

Punctuation carries useful structure, so it should be separated rather than discarded. Adding comma and period to the capturing group returns those delimiters as individual list items.

The split also creates empty strings where delimiters touch. We remove those artifacts in the following step.

In [4]:
# Capturing commas, periods, and whitespace keeps each delimiter in the result.
result = re.split(r"([,.]|\s)", text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


### Removing whitespace and empty fragments

The list comprehension keeps only items whose stripped value is non-empty. This removes separator whitespace and empty strings while preserving words and punctuation.

In [5]:
# Remove whitespace-only and empty fragments created by re.split.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


### Expanding the tokenization rule

Real text contains more than commas and periods. The next pattern handles several punctuation marks plus a double hyphen (`--`), then normalizes the result by stripping and filtering each fragment.

This is still a handcrafted tokenizer: it is useful for understanding the mechanics, but production tokenizers need broader rules and a strategy for unseen text.

In [6]:
# Expand the tokenizer to recognize common punctuation and double hyphens.
text = "Hello, world. Is this-- a test?"
result = re.split(r"([,.:;?_!\"()']|--|\s)", text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


## 2.3 Tokenizing the complete corpus

Now apply the same rule to the full story. The resulting `preprocessed` list is the ordered token sequence that can later be converted into token IDs.

Printing its length gives the corpus size under this particular tokenization scheme. A different tokenizer can produce a different token count for the same text.

In [7]:
# Apply the same tokenization rule to the complete corpus.
preprocessed = re.split(r"([,.:;?_!\"()']|--|\s)", raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


### Inspecting the token sequence

A short sample verifies that words and punctuation were separated as intended before vocabulary construction.

In [8]:
# Inspect the opening tokens to verify the preprocessing result.
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.4 Building a vocabulary

A **vocabulary** contains every distinct token the tokenizer knows. Converting `preprocessed` to a set removes duplicates; sorting makes the token-to-ID mapping deterministic and easier to inspect.

`vocab_size` is the number of unique tokens in this corpus, not the total number of token occurrences.

In [9]:
# A sorted set yields a deterministic list of unique corpus tokens.
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


### Mapping tokens to integer IDs

Neural networks operate on numbers rather than strings. `enumerate` assigns one integer ID to each token, producing the lookup dictionary `vocab`.

The sample output shows that punctuation and words occupy the same vocabulary and each receive their own ID.

In [10]:
# Assign a unique integer ID to every token.
vocab = {token: integer for integer, token in enumerate(all_words)}

# Display only the beginning of the vocabulary.
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


## 2.5 A simple text tokenizer

`SimpleTokenizerV1` stores mappings in both directions:

- `str_to_int` supports **encoding** text as token IDs.
- `int_to_str` supports **decoding** IDs back into readable text.

Encoding repeats the preprocessing rule used to build the vocabulary. Decoding joins the tokens and then removes spaces that were inserted before punctuation.

In [11]:
class SimpleTokenizerV1:
    """Encode known text tokens as IDs and decode IDs back to text."""

    def __init__(self, vocab: dict[str, int]) -> None:
        """Create forward and reverse lookup tables from a vocabulary."""
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        """Tokenize text and return the corresponding vocabulary IDs."""
        preprocessed = re.split(r"([,.?_!\"()']|--|\s)", text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        """Convert token IDs to text and restore punctuation spacing."""
        text = " ".join([self.int_to_str[i] for i in ids])

        # Remove spaces inserted immediately before punctuation.
        text = re.sub(r"\s+([,.?!\"()'])", r"\1", text)
        return text

### Encoding and decoding an in-vocabulary passage

This example uses text from the training corpus, so every token should exist in `vocab`. The encoded list is the numerical representation a language model would consume.

In [12]:
# Version 1 can encode passages containing only known vocabulary tokens.
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [13]:
# Decode the IDs to check that the text can be reconstructed.
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


### The out-of-vocabulary problem

A word that was absent from the corpus has no entry in `vocab`. Encoding such text with version 1 raises a `KeyError`. The commented call demonstrates this limitation without interrupting the notebook.

In [14]:
# "Hello" is absent from this corpus vocabulary, so version 1 raises KeyError.
text = "Hello, do you like tea?"
# print(tokenizer.encode(text)) # KeyError

## 2.6 Adding special context tokens

Special tokens let a tokenizer represent situations that ordinary corpus tokens cannot:

- `<|unk|>` replaces an unknown, out-of-vocabulary token.
- `<|endoftext|>` marks a document boundary, allowing multiple independent texts to be combined.

They are appended before rebuilding the vocabulary so they receive integer IDs like every other token.

In [15]:
# Add markers for document boundaries and unknown tokens.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


In [16]:
# Confirm that the special tokens were appended to the vocabulary.
for _, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Handling unknown tokens

`SimpleTokenizerV2` checks each parsed token during encoding and substitutes `<|unk|>` whenever the token is missing from the vocabulary. This makes encoding robust to new text while preserving the same decoding process.

In [17]:
class SimpleTokenizerV2:
    """A word-level tokenizer that replaces unseen tokens with <|unk|>."""

    def __init__(self, vocab: dict[str, int]) -> None:
        """Create forward and reverse lookup tables from a vocabulary."""
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        """Tokenize text, map unknown tokens, and return token IDs."""
        preprocessed = re.split(r"([,.:;?_!\"()']|--|\s)", text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Substitute the unknown-token marker before looking up IDs.
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        """Convert token IDs to text and restore punctuation spacing."""
        text = " ".join([self.int_to_str[i] for i in ids])

        # Remove spaces inserted immediately before punctuation.
        text = re.sub(r"\s+([,.:;?!\"()'])", r"\1", text)
        return text

### Joining independent documents

The end-of-text token separates passages that are not naturally continuous. The next cells verify that version 2 can encode new words as `<|unk|>`, retain the document boundary, and decode the complete sequence.

In [18]:
# Mark the boundary between two independent text samples.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [19]:
# Unknown words are mapped to <|unk|> instead of raising an error.
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [20]:
# Round-trip the joined documents through the tokenizer.
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 2.7 Byte pair encoding

The handcrafted tokenizers clarify the core ideas, but their fixed word vocabulary does not scale well. **Byte pair encoding (BPE)** builds a vocabulary of frequently occurring subword units. Common words can remain single tokens, while rare or invented words are decomposed into smaller pieces.

### Using `tiktoken`

`tiktoken` provides OpenAI's production-grade byte pair encoding implementations. The version check makes the environment explicit and helps reproduce the notebook.

In [21]:
# Report the installed tokenizer version for reproducibility.
from importlib.metadata import version

import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


### Loading the GPT-2 encoding

Unlike the word-level tokenizer above, GPT-2's BPE tokenizer can represent unfamiliar words by splitting them into known byte-based subword units. This avoids a dedicated unknown-word token for ordinary input.

In [22]:
# Load the byte pair encoding vocabulary used by GPT-2.
tokenizer = tiktoken.get_encoding("gpt2")

### Encoding text with BPE

`allowed_special` explicitly permits GPT-2's end-of-text marker. Notice that the example includes an invented place name: BPE can still encode it as a sequence of smaller known units.

In [23]:
# BPE can represent unfamiliar words as sequences of subword tokens.
text = "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [24]:
# Decode the BPE IDs to reconstruct the original string.
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## Exercise 2.1 — Byte pair encoding of unknown words

Encode an unfamiliar phrase and inspect both its token IDs and decoded pieces. The result demonstrates how BPE represents text that never appeared as a complete vocabulary entry.

In [25]:
# Exercise: inspect how GPT-2 tokenizes and reconstructs an unusual phrase.
weird_phrase = "Akwirw ier"
tokens = tokenizer.encode(weird_phrase, allowed_special={"<|endoftext|>"})
print(tokens)
strings = tokenizer.decode(tokens)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


## 2.8 Data sampling with a sliding window

A language model learns to predict the next token from the tokens that precede it. A sliding window turns one long token sequence
into many fixed-length training examples.

The next cell encodes the complete corpus with the GPT-2 tokenizer. The resulting list of token IDs is the source from which input
and target windows are sampled.

In [26]:
# Encode the complete corpus with GPT-2 BPE for window sampling.
with open("the-verdict.txt", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


### Selecting a demonstration sample

The first 50 token IDs are skipped only to make the decoded examples below more interesting. This is a presentation choice rather
than a required language-model preprocessing step.

In [27]:
# Skip the opening tokens to make the short demonstration more varied.
enc_sample = enc_text[50:]

### Creating shifted input-target windows

For a context size of four, `x` contains four consecutive token IDs. `y` contains the same sequence shifted one position to the
right. Each element in `y` is therefore the next-token target for the corresponding position in `x`.

In [28]:
# Shift the target window by one token relative to the input window.
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1 : context_size + 1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


### Visualizing next-token prediction

Growing the context one token at a time exposes the individual prediction tasks represented by a window. The first loop displays
raw token IDs; the following loop decodes the same relationships into readable text.

In [29]:
# Each prefix is context for predicting the token immediately after it.
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [30]:
# Decode the same pairs to make next-token prediction human-readable.
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


## 2.9 Packaging windows as a PyTorch dataset

`GPTDatasetV1` tokenizes a string and materializes overlapping fixed-length windows. Every target tensor contains the corresponding
input tensor shifted forward by one token.

- `max_length` controls the number of tokens in each example.
- `stride` controls how far the starting index moves between adjacent examples.
- `__getitem__` returns one `(input_ids, target_ids)` tensor pair.

The dataset loop uses `stride` as the third argument to `range`, so starting indices are `0, stride, 2 * stride, ...`. By contrast,
the `+1` in `target_chunk` creates next-token labels inside each example. These are independent shifts with different purposes.

The reusable class interface is fully typed, while obvious local variables remain unannotated because their types are inferable.

In [31]:
#| export
import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset


class GPTDatasetV1(Dataset):
    """Create overlapping input-target token windows from a text corpus."""

    def __init__(
        self,
        txt: str,
        tokenizer: tiktoken.Encoding,
        max_length: int,
        stride: int,
    ) -> None:
        """Tokenize text and materialize shifted windows as tensors.

        Args:
            txt: Complete text corpus used to create token windows.
            tokenizer: Tokenizer that converts the text into token IDs.
            max_length: Number of token IDs in each input and target window.
            stride: Number of token positions between consecutive window starts.
        """
        # Empty collections need annotations because their item type is not inferable.
        self.input_ids: list[torch.Tensor] = []
        self.target_ids: list[torch.Tensor] = []

        token_ids = tokenizer.encode(txt)

        # Stride is the range step: windows start at 0, stride, 2 * stride, and so on.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            # The separate +1 shift creates next-token targets within this window.
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self) -> int:
        """Return the number of available input-target windows.

        Returns:
            Number of input-target window pairs in the dataset.
        """
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Return one input window and its shifted target window.

        Args:
            idx: Zero-based index of the window pair to retrieve.

        Returns:
            Input and target token IDs, each shaped `(max_length,)`.
        """
        return self.input_ids[idx], self.target_ids[idx]

C:\Users\giloz\dev\build-llms-from-scratch-companion\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


### Creating a reusable data loader

`create_dataloader_v1` connects raw text, GPT-2 tokenization, window generation, and PyTorch batching behind one typed function.

`batch_size` controls examples per batch, `shuffle` randomizes their order, `drop_last` discards an incomplete final batch, and
`num_workers` controls background data-loading processes. The returned loader yields batches of input and target tensors.

The `#| export` directives make `GPTDatasetV1` and `create_dataloader_v1` reusable from the generated `data.py` module.

In [32]:
#| export
def create_dataloader_v1(
    txt: str,
    batch_size: int = 4,
    max_length: int = 256,
    stride: int = 128,
    shuffle: bool = True,
    drop_last: bool = True,
    num_workers: int = 0,
) -> DataLoader[tuple[torch.Tensor, torch.Tensor]]:
    """Build a data loader of shifted token windows from raw text.

    Args:
        txt: Complete text corpus used to create token windows.
        batch_size: Number of independent windows in each batch.
        max_length: Number of token IDs in each input and target window.
        stride: Number of token positions between consecutive window starts.
        shuffle: Whether to randomize window order before each iteration.
        drop_last: Whether to discard a final batch smaller than `batch_size`.
        num_workers: Number of subprocesses used to load batches.

    Returns:
        Data loader yielding `(inputs, targets)` tensors, each shaped
        `(batch_size, max_length)` for complete batches.
    """
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # DataLoader handles batching, optional shuffling, and worker processes.
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

### Inspecting the first generated window

Create batches containing one example and disable shuffling so the sampling order is deterministic. With `max_length=4`, each input
contains four tokens; with `stride=1`, the next dataset example begins only one token later and overlaps by three input tokens.

In [33]:
# Build one-example batches so each generated window is easy to inspect.
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False,
)

# With stride=1, consecutive examples begin one corpus token apart.
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


## Exercise 2.2 — Changing context size and stride

Run the data loader with combinations such as `max_length=2, stride=2` and `max_length=8, stride=2` to build intuition about
window sampling.

These parameters control different properties:

- `max_length` is the number of input tokens in each training example.
- `stride` is the distance between the starting positions of consecutive examples.
- The one-token offset between each input and target is fixed because the learning task is next-token prediction.

With `max_length=8` and `stride=2`, consecutive input windows overlap by six tokens. The two batches below make that overlap visible.

In [34]:
# Use an eight-token context while moving each new window by two tokens.
exo_2_2_dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=8,
    stride=2,
    shuffle=False,
)
exo_2_2_data_iter = iter(exo_2_2_dataloader)

# Since shuffling is disabled, these batches start at token offsets 0 and 2.
exo_2_2_first_batch = next(exo_2_2_data_iter)
print(exo_2_2_first_batch)
exo_2_2_first_batch = next(exo_2_2_data_iter)
print(exo_2_2_first_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


### Interpreting the exercise output

The first printed input window covers corpus offsets `0:8`; the second covers `2:10`. Their starting positions differ by the
configured stride of two, while each target is still its own input shifted by one token.

Changing to `max_length=2, stride=2` would produce starts at the same offsets but two-token windows, so adjacent input windows would
not overlap.

## 2.10 Inspecting batched input-target tensors

After studying individual windows, increase `batch_size` to group several examples into one tensor. Using `max_length=4` and
`stride=4` creates non-overlapping input windows because every new example begins four tokens after the previous one.

Within each example, the target still remains shifted by exactly one token. The stride changes sampling between examples; it does
not change the prediction offset between an input and its target.

In [35]:
# Use non-overlapping input windows: stride equals max_length.
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False,
)

# Unpack one batch into parallel input and next-token target tensors.
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:")
print(inputs)
print()
print("Targets:")
print(targets)

Inputs:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.11 Creating token embeddings

Token IDs are categorical labels; their numeric values do not express similarity or magnitude. An **embedding layer** maps each ID
to a dense vector whose values can be learned during model training.

For a vocabulary of size $V$ and embedding dimension $d$, the embedding table has shape $V 	imes d$. Passing a token ID to the
layer selects the corresponding row of that table.

### Preparing token IDs

`torch.nn.Embedding` expects integer indices. This short token sequence acts as a stand-in for IDs produced by the tokenizer and
data loader.

In [38]:
# Use a small sequence of token IDs to demonstrate embedding lookup.
input_ids = torch.tensor([2, 3, 5, 1])

### Choosing the embedding dimensions

`vocab_size` determines how many rows the embedding table needs, while `output_dim` determines the width of each token vector. The
tiny values below make the complete table easy to inspect.

In [36]:
# Six vocabulary entries become six rows in the embedding table.
vocab_size = 6
# Each token will be represented by a vector with three learned features.
output_dim = 3

### Initializing the embedding table

PyTorch initializes embedding weights randomly. Setting a manual seed makes the demonstration reproducible. The resulting weight
shape is `(vocab_size, output_dim)`, which is `(6, 3)` in this example.

In [41]:
# Fix the random seed so the initial embedding weights are reproducible.
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# The weight matrix has one output_dim-sized row per vocabulary token.
print(embedding_layer.weight)
print(embedding_layer.weight.shape)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
torch.Size([6, 3])


### Looking up one token

Calling the layer with token ID `3` retrieves the fourth row of the weight matrix because tensor indexing starts at zero. The output
keeps a leading sequence dimension, so one ID produces shape `(1, output_dim)`.

In [45]:
# Looking up token ID 3 selects row 3 from the embedding matrix.
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(torch.tensor([3])).shape)

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
torch.Size([1, 3])


### Embedding a token sequence

Applying the same layer to `input_ids` performs all row lookups at once. Four token IDs produce a tensor of shape `(4, 3)`: one
three-dimensional embedding for each token position. Repeated IDs would retrieve the same row until training updates the table.

In [43]:
# A sequence lookup returns one embedding vector for every input token ID.
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## 2.12 Encoding token positions

Token embeddings identify **what** each token represents, but they do not encode **where** the token appears. Without positional
information, the model would receive the same set of vectors for differently ordered sequences.

A learnable positional embedding table assigns one vector to every position in the context window. Adding token and positional
embeddings gives the model a representation containing both token identity and sequence location.

### Configuring the token embedding table

The number of rows must match the tokenizer vocabulary so every possible token ID has a vector. `output_dim` controls the number of
learned features in each vector and must also match the positional embedding width used later.

In [46]:
# GPT-2's tokenizer assigns IDs from a vocabulary of 50,257 entries.
vocab_size = 50257
# Use a compact embedding width for this learning example.
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

### Preparing a batch of token IDs

Create a deterministic batch with shape `(8, 4)`: eight training examples, each containing four token IDs. Setting `stride` equal
to `max_length` makes neighboring input windows non-overlapping.

In [47]:
# Create eight non-overlapping sequences containing four tokens each.
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False,
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

# The input shape is (batch_size, sequence_length).
print("Token IDs:")
print(inputs)
print()
print("Inputs shape:")
print(inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


### Embedding the complete batch

`torch.nn.Embedding` appends the embedding dimension to the input shape. An ID tensor with shape `(8, 4)` therefore becomes a token
embedding tensor with shape `(8, 4, 256)`.

In [51]:
# Replace every token ID with its output_dim-dimensional embedding vector.
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

### Creating learnable positional embeddings

`torch.arange(context_length)` produces position IDs `[0, 1, 2, 3]`. Looking them up yields one 256-dimensional vector per position,
so `pos_embeddings` has shape `(4, 256)`.

These are **absolute positional embeddings**: each vector represents a fixed index within the context window.

In [54]:
# Assign one learnable embedding vector to each position in the context window.
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
position_ids = torch.arange(context_length)
pos_embeddings = pos_embedding_layer(position_ids)
pos_embeddings.shape

torch.Size([4, 256])

### Combining token and positional embeddings with broadcasting

The token tensor has shape `(batch_size, context_length, output_dim)`, while the positional tensor has shape
`(context_length, output_dim)`.

PyTorch **broadcasts** the positional tensor across the missing batch dimension. Every sequence receives the same position-0 vector
at its first token, the same position-1 vector at its second token, and so on. The addition preserves the token tensor's shape.

In [56]:
# Broadcasting applies the same position vectors to every sequence in the batch.
input_embeddings = token_embeddings + pos_embeddings
input_embeddings.shape

torch.Size([8, 4, 256])

## Chapter 2 summary

This notebook has followed the text-processing pipeline from raw characters to order-aware model inputs:

1. load and inspect a text corpus;
2. split text into word and punctuation tokens;
3. build a vocabulary and token-to-ID mapping;
4. implement typed encoding and decoding interfaces;
5. handle unknown words and document boundaries with special tokens;
6. use GPT-2 byte pair encoding to represent arbitrary text;
7. create shifted input-target windows for next-token prediction;
8. control overlap between examples with the sliding-window stride;
9. package windows in a typed PyTorch dataset and data loader;
10. map token IDs to dense token embeddings; and
11. add positional embeddings so those vectors retain sequence order.

The combined input embeddings have shape `(batch_size, context_length, output_dim)` and are ready for the model's attention layers.